<a href="https://colab.research.google.com/github/fcoliveira-utfpr/aquacrop_ml/blob/main/01_vies_tecnologico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Início - Bibliotecas**
---

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# **Carregando Municípios**

---

In [2]:
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/produtividade_locais.csv"
df = pd.read_csv(url)

# Correção para o SyntaxError: remover o '.' final
cidade = df['Municipio']

df.drop(columns=['Municipio'], inplace=True)

# Identifica colunas que podem conter valores numéricos com vírgula como separador decimal
numeric_cols_potential = ['Latitude', 'Longitude', 'Altitude'] + [col for col in df.columns if str(col).isdigit()]

for col in numeric_cols_potential:
    if col in df.columns:
        # Substitui vírgulas por pontos e converte para numérico
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce') # 'coerce' transforma erros em NaN
df["Municipio"] = cidade
df_municipios = df
df_municipios

,Localidade,2007,2008,2009,2010,2011,2012,2013,2014,2015,...,2017,2018,2019,2020,2021,2022,Latitude,Longitude,Altitude,Municipio
0,Cascavel,2845.0,3246.639190,2566.822884,3350.942977,3517.101972,2670.760216,3593.658200,3740.810069,3753.398671,...,4032.132404,3558.264191,3328.905302,4302.280661,3630.562208,3157.191051,-25.03,-53.38,712,Cascavel
1,Foz do Iguaçu,3471.0,3482.411814,3493.898914,3403.459262,3642.748945,1562.603141,3920.076203,3605.711046,2927.753656,...,3526.565033,3086.249553,1561.400236,4543.559328,3637.898794,736.187120,-25.47,-54.48,275,Foz do Iguaçu
2,Guaíra,2836.0,2870.406280,1604.515952,3157.036693,3383.348742,1889.966974,3449.830267,2756.224765,2800.415370,...,3721.968373,3630.881827,2373.328358,4324.215085,3406.272279,806.650745,-24.19,-54.23,275,Guaíra
3,Marechal Cândido Rondon,2405.0,3248.645766,1709.202062,3180.265050,3343.830743,1384.688014,3559.996343,3244.423507,3029.418900,...,3895.660230,3754.331810,1806.019606,3885.526598,3458.676468,668.878584,-24.57,-54.14,327,Marechal Cândido Rondon
4,Matelândia,3100.0,3606.819496,1770.604491,3186.324621,3432.999562,1510.753590,3527.354543,3566.818903,2979.099739,...,4229.603503,4149.579231,2456.603037,4143.517208,3897.823571,1043.282319,-25.35,-53.91,324,Matelândia
5,Medianeira,3419.0,3477.395375,1250.193734,3625.643547,3744.077150,1640.885797,3658.941801,3729.551817,3125.949536,...,3846.033985,3859.108685,1989.223900,4402.552315,3897.823571,778.254956,-25.26,-54.09,401,Medianeira
6,Santa Helena,2844.0,3361.013995,1307.569775,3597.365547,3268.847871,1382.654699,2803.114599,2956.826344,3049.957333,...,3205.028321,3291.653725,1873.680283,3885.526598,3458.676468,759.324430,-24.89,-54.31,238,Santa Helena
7,São Miguel do Iguaçu,3248.0,3064.040818,1333.741302,3537.779761,3666.054432,1411.121119,3662.001969,3302.761721,2961.642071,...,3366.313617,3734.621308,1497.903293,4401.507818,3637.898794,820.322791,-25.39,-54.26,299,São Miguel do Iguaçu
8,São Pedro do Iguaçu,3069.0,3411.178383,2714.792674,3326.704691,3445.158947,1763.901399,3565.096625,3472.658978,3484.345196,...,4104.504011,3215.923904,2706.427075,4177.985589,3563.484846,1312.516466,-24.91,-53.89,517,São Pedro do Iguaçu
9,Toledo,3117.0,3174.402472,2278.936082,3438.806762,3394.494845,1383.671356,3654.861575,3556.584128,3644.544975,...,4238.908424,3475.272606,2175.550995,4177.985589,3511.080657,841.356709,-24.70,-53.78,524,Toledo


## **Correção do viés tecnológico**
---

In [3]:
# Lista de cidades da região Oeste do Paraná que serão analisadas
cidades = ['Cascavel',
 'Foz do Iguaçu',
 'Guaíra',
 'Marechal Cândido Rondon',
 'Matelândia',
 'Medianeira',
 'Santa Helena',
 'São Miguel do Iguaçu',
 'São Pedro do Iguaçu',
 'Toledo']

# URL do arquivo CSV contendo dados de produtividade de milho
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/produtividade_milho_oeste.csv"

# Carrega o arquivo CSV para um DataFrame do pandas
df = pd.read_csv(url)

# Filtra o DataFrame para manter apenas os dados das cidades listadas
df_filtrado = df[df['Localidade'].isin(cidades)]

# Remove as colunas 'Meedia' e 'Media' (provavelmente colunas com médias)
df_filtrado = df_filtrado.drop(columns=['Meedia', 'Media'])

# Transforma o DataFrame de formato largo para longo
# As colunas de anos se tornam uma única coluna 'Ano'
# Os valores de produtividade vão para a coluna 'Ya'
df_long = df_filtrado.melt(
    id_vars='Localidade',  # Mantém 'Localidade' como identificador
    var_name='Ano',        # Nome da nova coluna para os anos
    value_name='Ya'         # Nome da nova coluna para os valores
)

# Converte a coluna 'Ano' para o tipo inteiro
df_long['Ano'] = df_long['Ano'].astype(int)

# Prepara os dados para regressão linear
# X: variável independente (Ano)
# y: variável dependente (Ya - produtividade)
X = df_long[['Ano']]
y = df_long['Ya']

# Cria e treina um modelo de regressão linear
model = LinearRegression()
model.fit(X, y)

# Extrai os coeficientes do modelo treinado
coef = model.coef_[0]      # Coeficiente angular (inclinação)
intercept = model.intercept_ # Coeficiente linear (intercepto)
r2 = model.score(X, y)      # Coeficiente de determinação R²

# Calcula a produtividade máxima (Ym) para cada linha
df_long["Ym"] = df_long['Ya'].max()

# Calcula o déficit relativo de produtividade: 1 - (Ya/Ym)
df_long["1-(Ya/Ym)"] = 1 - (df_long['Ya'] / df_long['Ym'])

# Calcula a produtividade prevista pelo modelo de regressão linear (Y)
df_long["Y"] = coef * df_long['Ano'] + intercept

# Calcula o desvio relativo (RD): diferença entre real e previsto dividido pelo previsto
df_long["RD"] = (df_long["Ya"] - df_long["Y"]) / df_long["Y"]

# Calcula a produtividade ajustada (AY):
# (RD + 1) multiplicado pela máxima produtividade prevista
df_long["AY"] = (df_long["RD"] + 1) * df_long["Y"].max()

# Transforma os dados de volta para formato largo
# Cada ano se torna uma coluna com os valores de AY
df_ay = df_long.pivot(
    index='Localidade',     # Linhas serão as localidades
    columns='Ano',          # Colunas serão os anos
    values='AY'             # Valores serão a produtividade ajustada
).reset_index()

# Obtém dados geográficos dos municípios
lat = df_municipios['Latitude']
lon = df_municipios['Longitude']
alt = df_municipios['Altitude']
muni = df_municipios['Municipio']

# Adiciona as colunas de dados geográficos ao DataFrame
df_ay['Latitude'] = lat
df_ay['Longitude'] = lon
df_ay['Altitude'] = alt
df_ay['Municipio'] = muni

# Operações de limpeza (mas sem atribuição, então não alteram o DataFrame)
df_ay.reset_index()        # Reseta o índice (mas não salva)
df_ay.drop(columns=['Localidade'])  # Tenta remover Localidade (mas não salva)

# Exibe o DataFrame final
df_ay

Ano,Localidade,2007,2008,2009,2010,2011,2012,2013,2014,2015,...,2017,2018,2019,2020,2021,2022,Latitude,Longitude,Altitude,Municipio
0,Cascavel,2845.0,3246.639190,2566.822884,3350.942977,3517.101972,2670.760216,3593.658200,3740.810069,3753.398671,...,4032.132404,3558.264191,3328.905302,4302.280661,3630.562208,3157.191051,-25.03,-53.38,712,Cascavel
1,Foz do Iguaçu,3471.0,3482.411814,3493.898914,3403.459262,3642.748945,1562.603141,3920.076203,3605.711046,2927.753656,...,3526.565033,3086.249553,1561.400236,4543.559328,3637.898794,736.187120,-25.47,-54.48,275,Foz do Iguaçu
2,Guaíra,2836.0,2870.406280,1604.515952,3157.036693,3383.348742,1889.966974,3449.830267,2756.224765,2800.415370,...,3721.968373,3630.881827,2373.328358,4324.215085,3406.272279,806.650745,-24.19,-54.23,275,Guaíra
3,Marechal Cândido Rondon,2405.0,3248.645766,1709.202062,3180.265050,3343.830743,1384.688014,3559.996343,3244.423507,3029.418900,...,3895.660230,3754.331810,1806.019606,3885.526598,3458.676468,668.878584,-24.57,-54.14,327,Marechal Cândido Rondon
4,Matelândia,3100.0,3606.819496,1770.604491,3186.324621,3432.999562,1510.753590,3527.354543,3566.818903,2979.099739,...,4229.603503,4149.579231,2456.603037,4143.517208,3897.823571,1043.282319,-25.35,-53.91,324,Matelândia
5,Medianeira,3419.0,3477.395375,1250.193734,3625.643547,3744.077150,1640.885797,3658.941801,3729.551817,3125.949536,...,3846.033985,3859.108685,1989.223900,4402.552315,3897.823571,778.254956,-25.26,-54.09,401,Medianeira
6,Santa Helena,2844.0,3361.013995,1307.569775,3597.365547,3268.847871,1382.654699,2803.114599,2956.826344,3049.957333,...,3205.028321,3291.653725,1873.680283,3885.526598,3458.676468,759.324430,-24.89,-54.31,238,Santa Helena
7,São Miguel do Iguaçu,3248.0,3064.040818,1333.741302,3537.779761,3666.054432,1411.121119,3662.001969,3302.761721,2961.642071,...,3366.313617,3734.621308,1497.903293,4401.507818,3637.898794,820.322791,-25.39,-54.26,299,São Miguel do Iguaçu
8,São Pedro do Iguaçu,3069.0,3411.178383,2714.792674,3326.704691,3445.158947,1763.901399,3565.096625,3472.658978,3484.345196,...,4104.504011,3215.923904,2706.427075,4177.985589,3563.484846,1312.516466,-24.91,-53.89,517,São Pedro do Iguaçu
9,Toledo,3117.0,3174.402472,2278.936082,3438.806762,3394.494845,1383.671356,3654.861575,3556.584128,3644.544975,...,4238.908424,3475.272606,2175.550995,4177.985589,3511.080657,841.356709,-24.70,-53.78,524,Toledo


In [6]:
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/produtividade_locais.csv"
df = pd.read_csv(url)
#df.to_excel("produtividade_locais_ajustado.xlsx", index=False)
#